# Session 11: Enterprise Security Tooling & Automation

> Hands-on with SAST/DAST tools, vulnerability triage, and AI-assisted patch generation.

## 1. SAST vs DAST

Security tooling falls into two broad categories that complement each other — neither is sufficient alone.

**SAST (Static Application Security Testing)** analyses your source code or compiled artefacts *without running the application*. It works like a very security-aware linter: it scans for patterns that are known to produce vulnerabilities (e.g., calls to `exec()` with user input, SQL query concatenation, use of `MD5` for password hashing). Because it runs on code, it can be integrated into your IDE and CI pipeline to give feedback before a single request is ever made. The downside is **false positives** — it may flag safe code as dangerous — and it cannot detect vulnerabilities that only appear at runtime (e.g., a broken access control that depends on how two services interact).

**DAST (Dynamic Application Security Testing)** tests a *running application* by sending it crafted HTTP requests — fuzzing inputs, probing for SQL injection, trying XSS payloads, testing authentication bypasses. It finds issues that only manifest when the full stack is live. The downside is that it requires a running environment, is slower to run, and may miss vulnerabilities in code paths that are hard to reach through HTTP.

**How they fit in a CI/CD pipeline:**
- SAST runs on every commit (fast, catches obvious issues early).
- DAST runs in a staging environment after deployment (slower, catches runtime issues).
- Neither replaces manual security review or penetration testing for critical systems.

| | SAST | DAST |
|---|---|---|
| When | Before runtime | Against a running app |
| Finds | Code-level bugs | Runtime/integration bugs |
| Speed | Fast (seconds–minutes) | Slow (minutes–hours) |
| Example tools | Bandit, Semgrep, SonarQube | OWASP ZAP, Burp Suite |

| | SAST (Static) | DAST (Dynamic) |
|--|---------------|----------------|
| **When** | At commit / CI gate | Against running app |
| **Finds** | Code vulnerabilities | Runtime vulnerabilities |
| **Tools** | Semgrep, Bandit, SonarQube | OWASP ZAP, Burp Suite |
| **Speed** | Fast (seconds) | Slow (minutes-hours) |
| **False Positives** | Higher | Lower |

**Strategy:** SAST as pre-commit + CI gate → DAST in staging pipeline

## 2. Vulnerability Triage Framework

When a security scanner runs against your codebase, it will typically report dozens or hundreds of findings — many of which are low-severity or false positives. Without a triage framework, teams either ignore everything (dangerous) or try to fix everything at once (paralysing).

**Triage is the process of prioritising which vulnerabilities to fix first, based on actual risk.**

The standard scoring system is **CVSS (Common Vulnerability Scoring System)**, which produces a score from 0–10 based on:
- **Attack Vector:** Can it be exploited remotely? (Network > Adjacent > Local > Physical)
- **Attack Complexity:** How hard is it to exploit?
- **Privileges Required / User Interaction:** Does the attacker need an account? Does a victim need to click something?
- **Impact (CIA triad):** How severe is the effect on Confidentiality, Integrity, and Availability?

**Practical triage rules:**
1. **Critical (9.0–10):** Fix immediately. These are remotely exploitable with no authentication required. Drop everything.
2. **High (7.0–8.9):** Fix in the current sprint. Schedule a patch release.
3. **Medium (4.0–6.9):** Schedule for the next sprint. Assess whether a workaround exists in the meantime.
4. **Low / Informational:** Add to the backlog. Review quarterly.

**False positive management:** Before fixing, verify that the vulnerability is actually exploitable in your context. A SAST tool may flag `os.system(cmd)` — but if `cmd` is always a hardcoded string constant, the risk is zero. Document the rationale for accepted false positives so future engineers understand why they were not fixed.

In [ ]:
# Vulnerability Severity Classification
vulnerabilities = [
    {'id':'CVE-001','type':'SQL Injection','cvss':9.8,'exploitable':True},
    {'id':'CVE-002','type':'XSS Stored','cvss':7.5,'exploitable':True},
    {'id':'CVE-003','type':'Outdated Dependency','cvss':4.2,'exploitable':False},
    {'id':'CVE-004','type':'Missing Rate Limit','cvss':5.3,'exploitable':True},
]

def triage(vuln):
    if vuln['cvss'] >= 9.0 or (vuln['cvss'] >= 7.0 and vuln['exploitable']):
        return 'P0 — Fix immediately, block release'
    elif vuln['cvss'] >= 5.0:
        return 'P1 — Fix within current sprint'
    return 'P2 — Fix in next sprint'

for v in sorted(vulnerabilities, key=lambda x: x['cvss'], reverse=True):
    print(f"{v['id']} [{v['type']}] CVSS:{v['cvss']} → {triage(v)}")

## 💡 Additional Examples: Security Tooling

**Example 1 — SAST in CI: blocking merges on High findings**
A SAST tool integrated into a CI pipeline as a blocking step means that no PR can be merged if it introduces a High or Critical finding. This shifts security left — the developer gets feedback in their PR review, not six months later in a penetration test. The key to making this work is low false-positive rate: if the scanner generates too much noise, developers will learn to ignore it.

**Example 2 — Dependency scanning: SCA (Software Composition Analysis)**
Beyond your own code, your dependencies are an attack surface. A library you installed in 2022 may have received a critical CVE in 2024. Tools like `pip-audit`, `safety`, Dependabot, and Snyk continuously monitor your `requirements.txt` (or `package.json`) against known vulnerability databases. Automating this as a weekly CI job or using GitHub Dependabot ensures you're notified before a patched version is exploited in the wild.

**Example 3 — Secrets scanning: prevent commits before they reach remote**
Tools like `truffleHog`, `gitleaks`, and `detect-secrets` can be run as a **pre-commit hook** to scan staged files for high-entropy strings and known credential patterns before they ever reach the remote repository. This is the last line of defence before a secret enters git history. Once it's in git history, even a `git rebase` doesn't fully erase it — you must rotate the secret and assume it is compromised.

In [ ]:
# ─── Example 1: Bandit SAST — detecting dangerous code patterns ──────────────
# Simulates how Bandit analyses common vulnerable patterns

vulnerable_code_samples = [
    {
        'id': 'B101',
        'check': 'assert_used',
        'code': 'assert user.is_admin, "Must be admin"',
        'severity': 'LOW',
        'confidence': 'HIGH',
        'issue': 'Use of assert detected. Assertions can be disabled with -O flag.',
        'fix': 'Use explicit if-raise pattern instead:\n'
               '  if not user.is_admin:\n'
               '      raise PermissionError("Must be admin")',
    },
    {
        'id': 'B105',
        'check': 'hardcoded_password_string',
        'code': 'password = "admin123"',
        'severity': 'MEDIUM',
        'confidence': 'MEDIUM',
        'issue': 'Possible hardcoded password.',
        'fix': 'Use environment variable:\n'
               '  password = os.getenv("DB_PASSWORD")',
    },
    {
        'id': 'B602',
        'check': 'subprocess_popen_with_shell_equals_true',
        'code': 'subprocess.Popen(user_input, shell=True)',
        'severity': 'HIGH',
        'confidence': 'HIGH',
        'issue': 'subprocess call with shell=True — command injection risk!',
        'fix': 'Use list form without shell=True:\n'
               '  subprocess.Popen(["safe_cmd", "--arg", validated_value])',
    },
    {
        'id': 'B608',
        'check': 'hardcoded_sql_expressions',
        'code': "query = f\"SELECT * FROM users WHERE id={user_id}\"",
        'severity': 'MEDIUM',
        'confidence': 'MEDIUM',
        'issue': 'Possible SQL injection via string formatting.',
        'fix': 'Use parameterized queries:\n'
               '  cursor.execute("SELECT * FROM users WHERE id = %s", (user_id,))',
    },
]

SEVERITY_COLORS = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}

print('=== BANDIT SAST Report ===\n')
for finding in vulnerable_code_samples:
    icon = SEVERITY_COLORS[finding['severity']]
    print(f'{icon} [{finding["id"]}] {finding["check"]}')
    print(f'   Severity: {finding["severity"]} | Confidence: {finding["confidence"]}')
    print(f'   Code:     {finding["code"]}')
    print(f'   Issue:    {finding["issue"]}')
    print(f'   Fix:\n' + '\n'.join(f'      {l}' for l in finding['fix'].split('\n')))
    print()

high_count = sum(1 for f in vulnerable_code_samples if f['severity'] == 'HIGH')
print(f'Summary: {len(vulnerable_code_samples)} issues found | {high_count} HIGH severity → BLOCK CI pipeline')

# ─── Example 2: Dependency Vulnerability Scanner ─────────────────────────────
# Simulates output from `pip-audit` / `safety check`
known_cves = {
    'django': {
        '3.2.0': [{'cve': 'CVE-2023-24580', 'cvss': 7.5, 'desc': 'Denial-of-service via file upload'}],
        '4.1.0': [{'cve': 'CVE-2023-43665', 'cvss': 7.5, 'desc': 'Denial-of-service in Truncator'}],
    },
    'pillow': {
        '9.0.0': [{'cve': 'CVE-2023-44271', 'cvss': 7.5, 'desc': 'Uncontrolled memory growth via ImageFont'}],
    },
    'cryptography': {
        '41.0.0': [{'cve': 'CVE-2023-49083', 'cvss': 4.0, 'desc': 'NULL pointer dereference in PKCS12'}],
    },
}

installed_packages = {
    'django': '3.2.0',
    'pillow': '9.0.0',
    'cryptography': '41.0.0',
    'requests': '2.31.0',     # Safe version
    'numpy': '1.26.0',        # No known CVEs
}

print('\n=== Dependency Vulnerability Scan ===')
print(f'{"Package":<20} {"Version":<12} {"Status":<10} CVEs')
print('-' * 65)

vulnerable_deps = []
for package, version in installed_packages.items():
    pkg_cves = known_cves.get(package, {}).get(version, [])
    if pkg_cves:
        cve_list = ', '.join(f'{c["cve"]}(CVSS:{c["cvss"]})' for c in pkg_cves)
        print(f'{"⚠️  " + package:<20} {version:<12} {"VULNERABLE":<10} {cve_list}')
        vulnerable_deps.append({'package': package, 'cves': pkg_cves})
    else:
        print(f'{"✅  " + package:<20} {version:<12} {"OK":<10}')

if vulnerable_deps:
    print(f'\n🚨 Found {len(vulnerable_deps)} vulnerable package(s)!')
    print('Action: Run `pip install --upgrade ' + ' '.join(d["package"] for d in vulnerable_deps) + '`')

# ─── Example 3: Security Headers Checker (HTTP Response) ─────────────────────
def check_security_headers(response_headers: dict) -> dict:
    """
    Validate HTTP response headers against OWASP best practices.
    Mirrors the behaviour of DAST security header scanner tools.
    """
    required_headers = {
        'Strict-Transport-Security': {
            'recommended': 'max-age=31536000; includeSubDomains',
            'severity': 'HIGH',
            'reason': 'HSTS prevents protocol downgrade attacks',
        },
        'X-Content-Type-Options': {
            'recommended': 'nosniff',
            'severity': 'MEDIUM',
            'reason': 'Prevents MIME type sniffing attacks',
        },
        'X-Frame-Options': {
            'recommended': 'DENY',
            'severity': 'MEDIUM',
            'reason': 'Prevents clickjacking attacks',
        },
        'Content-Security-Policy': {
            'recommended': "default-src 'self'; script-src 'self' 'nonce-{random}'",
            'severity': 'HIGH',
            'reason': 'Primary defense against XSS attacks',
        },
        'Referrer-Policy': {
            'recommended': 'strict-origin-when-cross-origin',
            'severity': 'LOW',
            'reason': 'Controls referrer information leakage',
        },
    }

    results = {'passed': [], 'failed': []}
    for header, meta in required_headers.items():
        if header in response_headers:
            results['passed'].append({'header': header, 'value': response_headers[header]})
        else:
            results['failed'].append({**meta, 'header': header})
    return results

# Simulate response headers from two servers
good_headers = {
    'Strict-Transport-Security': 'max-age=31536000; includeSubDomains',
    'X-Content-Type-Options': 'nosniff',
    'X-Frame-Options': 'DENY',
    'Content-Security-Policy': "default-src 'self'",
    'Referrer-Policy': 'strict-origin-when-cross-origin',
}
bad_headers = {'Content-Type': 'application/json', 'Server': 'nginx/1.18.0'}  # No security headers!

print('\n=== Security Header Scan ===')
for label, headers in [('SECURE server', good_headers), ('INSECURE server', bad_headers)]:
    result = check_security_headers(headers)
    print(f'\n{label}:')
    for p in result['passed']:  print(f'  ✅ {p["header"]}: {p["value"][:50]}')
    for f in result['failed']:  print(f'  ❌ [{f["severity"]}] Missing {f["header"]} — {f["reason"]}')


## 3. AI Lab: AI Security Auditor

### 🧪 AI Lab / Practice

> **TODO:** Run Bandit (Python SAST) on your project: `bandit -r ./src -f json > report.json` → Feed the report to Claude: 'Analyze this SAST report. For each finding, provide: root cause, fix with code example, and whether it is a false positive. Prioritize by exploitability.' → Implement all P0 and P1 fixes.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')